In [1]:
print (ord('杨'))

26472


In [2]:
print (chr(26472))

杨


In [3]:
print (repr(chr(0)))

'\x00'


In [4]:
print("this is a test" + chr(0) + "string")

this is a test string


In [5]:
test_string = "hello! こんにちは!"
utf8_encoded = test_string.encode("utf-8")
print (utf8_encoded)

b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'


In [6]:
print (type(utf8_encoded))

<class 'bytes'>


In [7]:
# 打印出来来是10进制数字数组
print (list(utf8_encoded))

[104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33]


怎么算出来的？

十六进制是 16进制，所以：

e3 = e × 16¹ + 3 × 16⁰

其中：

e = 14（因为 a=10, b=11, ..., e=14）

所以：

= 14 × 16 + 3
= 224 + 3
= 227

In [8]:
print (len(utf8_encoded))

23


In [9]:
print (utf8_encoded.decode("utf-8"))

hello! こんにちは!


In [10]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

In [11]:
decode_utf8_bytes_to_str_wrong("HELLO".encode("utf-8"))

'HELLO'

In [12]:
decode_utf8_bytes_to_str_wrong("你是谁".encode("utf-8"))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe4 in position 0: unexpected end of data

# BPE Training Example

In [1]:
# 论文原始 toy vocab（词已经被拆成字符序列）
vocab = {
    'l o w </w>'      : 5,
    'l o w e r </w>'  : 2,
    'n e w e s t </w>': 6,
    'w i d e s t </w>': 3,
}

In [14]:
for word, freq in vocab.items():
    print (f"freq={freq}, word_repr='{word}'")

freq=5, word_repr='l o w </w>'
freq=2, word_repr='l o w e r </w>'
freq=6, word_repr='n e w e s t </w>'
freq=3, word_repr='w i d e s t </w>'


## 实现get_stats

In [2]:
import collections

def get_stats(vocab):
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

In [3]:
pairs = get_stats(vocab)

In [17]:
for pair, freq in sorted(pairs.items(), key=lambda x: -x[1]):
    print(f"freq={freq}, pair={pair}")

freq=9, pair=('e', 's')
freq=9, pair=('s', 't')
freq=9, pair=('t', '</w>')
freq=8, pair=('w', 'e')
freq=7, pair=('l', 'o')
freq=7, pair=('o', 'w')
freq=6, pair=('n', 'e')
freq=6, pair=('e', 'w')
freq=5, pair=('w', '</w>')
freq=3, pair=('w', 'i')
freq=3, pair=('i', 'd')
freq=3, pair=('d', 'e')
freq=2, pair=('e', 'r')
freq=2, pair=('r', '</w>')


## 实现merge_vocab

In [4]:
import re

def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

In [20]:
best = max(pairs, key=pairs.get)

In [21]:
print(f"best pair: {best}")

best pair: ('e', 's')


In [22]:
vocab = merge_vocab(best, vocab)

In [23]:
for word, freq in vocab.items():
    print(f"freq={freq}, word_repr='{word}'")

freq=5, word_repr='l o w </w>'
freq=2, word_repr='l o w e r </w>'
freq=6, word_repr='n e w es t </w>'
freq=3, word_repr='w i d es t </w>'


## 主循环

In [5]:
# 训练，同时保存 merge rules
num_merges = 10
merge_rules = []                          # ← 新增：记录每轮的 best pair

for i in range(num_merges):
    pairs = get_stats(vocab)
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    merge_rules.append(best)              # ← 新增

print("学到的 merge rules（顺序很重要）：")
for i, rule in enumerate(merge_rules):
    print(f"  {i+1:>2}. {rule[0]} + {rule[1]} → {''.join(rule)}")

学到的 merge rules（顺序很重要）：
   1. e + s → es
   2. es + t → est
   3. est + </w> → est</w>
   4. l + o → lo
   5. lo + w → low
   6. n + e → ne
   7. ne + w → new
   8. new + est</w> → newest</w>
   9. low + </w> → low</w>
  10. w + i → wi


## 实现encode

In [6]:
def encode(word, merge_rules):
    # 第一步: 拆成字符，词尾加 </w>
    symbols = list(word) + ['</w>']

    # 第二步: 按顺序应用每条 merge rule
    for rule in merge_rules:
        i = 0
        while i < len(symbols) - 1:
            if symbols[i] == rule[0] and symbols[i+1] == rule[1]:
                symbols = symbols[:i] + [''.join(rule)] + symbols[i+2:]
            else:
                i += 1
    return symbols

In [7]:
for test_word in ['low', 'lower', 'newest', 'widest']:
    tokens = encode(test_word, merge_rules)
    print(f"{test_word:>10} → {tokens}")

       low → ['low</w>']
     lower → ['low', 'e', 'r', '</w>']
    newest → ['newest</w>']
    widest → ['wi', 'd', 'est</w>']


In [8]:
for test_word in ['newer', 'lowest', 'wild']:
    tokens = encode(test_word, merge_rules)
    print(f"{test_word:>10} → {tokens}")

     newer → ['new', 'e', 'r', '</w>']
    lowest → ['low', 'est</w>']
      wild → ['wi', 'l', 'd', '</w>']


In [9]:
print(merge_rules)

[('e', 's'), ('es', 't'), ('est', '</w>'), ('l', 'o'), ('lo', 'w'), ('n', 'e'), ('ne', 'w'), ('new', 'est</w>'), ('low', '</w>'), ('w', 'i')]


## 完整算法

In [13]:
# 论文原始 toy vocab（词已经被拆成字符序列）
vocab = {
    'l o w </w>'      : 5,
    'l o w e r </w>'  : 2,
    'n e w e s t </w>': 6,
    'w i d e s t </w>': 3,
}

import collections

def get_stats(vocab):
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

import re

def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

# 训练，同时保存 merge rules
num_merges = 12
merge_rules = []                          # ← 新增：记录每轮的 best pair

for i in range(num_merges):
    pairs = get_stats(vocab)
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    merge_rules.append(best)              # ← 新增

print("学到的 merge rules（顺序很重要）：")
for i, rule in enumerate(merge_rules):
    print(f"  {i+1:>2}. {rule[0]} + {rule[1]} → {''.join(rule)}")

学到的 merge rules（顺序很重要）：
   1. e + s → es
   2. es + t → est
   3. est + </w> → est</w>
   4. l + o → lo
   5. lo + w → low
   6. n + e → ne
   7. ne + w → new
   8. new + est</w> → newest</w>
   9. low + </w> → low</w>
  10. w + i → wi
  11. wi + d → wid
  12. wid + est</w> → widest</w>


In [14]:
def encode(word, merge_rules):
    # 第一步: 拆成字符，词尾加 </w>
    symbols = list(word) + ['</w>']

    # 第二步: 按顺序应用每条 merge rule
    for rule in merge_rules:
        i = 0
        while i < len(symbols) - 1:
            if symbols[i] == rule[0] and symbols[i+1] == rule[1]:
                symbols = symbols[:i] + [''.join(rule)] + symbols[i+2:]
            else:
                i += 1
    return symbols

In [15]:
for test_word in ['newer', 'lowest', 'wild']:
    tokens = encode(test_word, merge_rules)
    print(f"{test_word:>10} → {tokens}")

for test_word in ['low', 'lower', 'newest', 'widest']:
    tokens = encode(test_word, merge_rules)
    print(f"{test_word:>10} → {tokens}")

     newer → ['new', 'e', 'r', '</w>']
    lowest → ['low', 'est</w>']
      wild → ['wi', 'l', 'd', '</w>']
       low → ['low</w>']
     lower → ['low', 'e', 'r', '</w>']
    newest → ['newest</w>']
    widest → ['widest</w>']


## 完整的BPE实现

In [17]:
import re, collections

class BPETokenizer:
    def __init__(self, num_merges):
        self.num_merges = num_merges
        self.merge_rules = []
        self.vocab = {}

    def train(self, corpus):
        """从原始文本语料训练，建立初始字符词表并学习 merge rules"""
        # 统计词频，初始化为字符序列
        word_freq = collections.defaultdict(int)
        for word in corpus.split():
            word_freq[word] += 1

        self.vocab = {
            ' '.join(list(word)) + ' </w>': freq
            for word, freq in word_freq.items()
        }

        # 训练循环
        for i in range(self.num_merges):
            pairs = self._get_stats()
            if not pairs:
                break
            best = max(pairs, key=pairs.get)
            self._merge_vocab(best)
            self.merge_rules.append(best)

        return self

    def encode(self, word):
        """把一个词编码为 subword token 列表"""
        symbols = list(word) + ['</w>']
        for rule in self.merge_rules:
            i = 0
            while i < len(symbols) - 1:
                if symbols[i] == rule[0] and symbols[i+1] == rule[1]:
                    symbols = symbols[:i] + [''.join(rule)] + symbols[i+2:]
                else:
                    i += 1
        return symbols

    def tokenize(self, text):
        """对整段文本分词，返回所有 token"""
        tokens = []
        for word in text.split():
            tokens.extend(self.encode(word))
        return tokens

    def _get_stats(self):
        pairs = collections.defaultdict(int)
        for word, freq in self.vocab.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pairs[symbols[i], symbols[i+1]] += freq
        return pairs

    def _merge_vocab(self, pair):
        new_vocab = {}
        bigram = re.escape(' '.join(pair))
        p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
        for word in self.vocab:
            w_out = p.sub(''.join(pair), word)
            new_vocab[w_out] = self.vocab[word]
        self.vocab = new_vocab


# ── 测试 ──────────────────────────────────────────────
corpus = "low low low low low lower lower newest newest newest newest newest newest widest widest widest"

tokenizer = BPETokenizer(num_merges=3)
tokenizer.train(corpus)

print("── merge rules ──")
for i, rule in enumerate(tokenizer.merge_rules):
    print(f"  {i+1:>2}. {rule[0]} + {rule[1]} → {''.join(rule)}")

print("\n── encode ──")
for word in ['low', 'lower', 'newest', 'widest', 'newer', 'lowest', 'wild']:
    print(f"  {word:>10} → {tokenizer.encode(word)}")

print("\n── tokenize ──")
text = "the newest low widest"
print(f"  input : '{text}'")
print(f"  tokens: {tokenizer.tokenize(text)}")

── merge rules ──
   1. e + s → es
   2. es + t → est
   3. est + </w> → est</w>

── encode ──
         low → ['l', 'o', 'w', '</w>']
       lower → ['l', 'o', 'w', 'e', 'r', '</w>']
      newest → ['n', 'e', 'w', 'est</w>']
      widest → ['w', 'i', 'd', 'est</w>']
       newer → ['n', 'e', 'w', 'e', 'r', '</w>']
      lowest → ['l', 'o', 'w', 'est</w>']
        wild → ['w', 'i', 'l', 'd', '</w>']

── tokenize ──
  input : 'the newest low widest'
  tokens: ['t', 'h', 'e', '</w>', 'n', 'e', 'w', 'est</w>', 'l', 'o', 'w', '</w>', 'w', 'i', 'd', 'est</w>']
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
